# 3.7. Softmax Regression
D2L의 Softmax Regression장을 PyTorch 기준으로 정리함.

이장은 지금까지 배운 선형회귀를 숫자예측에서 카테고리 예측으로 확장한 것이라고 보면 된다.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [ ]:
%matplotlib inline
import random
import torch
import matplotlib.pyplot as plt
from torch.distributions.multinomial import Multinomial
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

torch.manual_seed(0)
random.seed(0)

print("PyTorch version:", torch.__version__)

## 1. 회귀와 분류는 무엇이 다른가?

선형회귀는 숫자 하나를 예측했었다.

예를 들어서  
> 집 가격은 얼마인가? 환자의 입원 기간은 며칠인가?
이런것이다.

분류는 여러 클래스 중에서 정답을 고르는 것이다.

예를 들어서
> 이 이미지는 고양이인가, 닭인가, 개인가? 이 이메일은 스팸인가, 정상인가?
이런것이다.

D2L은 이장은 하나의 데이터가 여러 클래스 중 정확히 하나의 클래스에 속한다고 가정하는 다중 클래스 분류를 다룬다.

## 2. 예제: 2x2 이미지 분류

2x2흑백 이미지가 있다고 할때

10 30  
20 50

이걸 픽셀을 펼치면 feature 4개가 된다.
```py
x = [10, 30, 20, 50]

x.shape == 4 # shape는 4일 것이다.
```
분류할 클래스를 3개로 가정할때

```text
0: 고양이
1: 닭
2: 개
```

입력 feature는 4개이고, 출력 클래스는 3개이다. 따라서 모델은 하나의 숫자가 아니라 클래스마다 하나씩 총 3개의 숫자를 출력해야한다.

## 3. 라벨은 어떻게 표현하나?

### 정수 라벨
컴퓨터에서는 보통 다음처럼 저장한다.

```py
고양이 = 0
닭 = 1
개 = 2

y = 2 # 예를 들어서 정답이 개라면 이렇다.
```
하지만 여기서 2가 고양이 0보다 더 크다는 의미는 아니다. 단순 클래스 이름 대신 붙혀서 쓰는 번호이다.

### One-hot encoding
수학적으로는 클래스를 one-hot vector로 표현할 수 있다.

```
고양이: [1, 0, 0]
닭:     [0, 1, 0]
개:     [0, 0, 1]
```
여기서 만약 정답이 개라면
```py
y_one_hot = [0, 0, 1]
```

정답 위치만 1이고 나머지는 0이다.D2L은 cross-entropy 수식을 설명하기 위해 주로 one-hot표현을 쓴다.

하지만 PyTorch의 nn.CrossEntropyLoss에선 보통 one-hot이 아니라 정수 클래스 번호를 넣는다고 한다.

```py
y = torch.tensor([2]) # 개
```
PyTorch의 CrossEntropyLoss는 입력으로 logits를 받고, target은 일반적으로 0부터 C-1 사이의 클래스 인덱스를 받는다고 한다.

## 4. 선형회귀에서는 출력이 하나

기존 선형회귀는 다음과 같다.

$$
\hat{y}=Xw+b
$$

예를 들어서 feature가 4개면
```py
X.shape == [batch_size, 4]
w.shape == [4, 1]
b.shape == [1]

y_hat = X @ w + b
y_hat.shape == [batch_size, 1]
```
출력이 하나인 이유는 집 가격처럼 숫자 하나를 예측하기 때문이다.

## 5. 분류에서는 클래스마다 출력이 필요하다

클래스가 만약 고양이, 닭, 개 3개라면 각 클래스에 대한 점수가 필요하다 / 고양이 점수, 닭 점수, 개 점수

그래서 가중치도 클래스마다 따로 존재한다.

$$
o_1=x_1w_{11}+x_2w_{21}+x_3w_{31}+x_4w_{41}+b_1\\
o_2=x_1w_{12}+x_2w_{22}+x_3w_{32}+x_4w_{42}+b_2\\
o_3=x_1w_{13}+x_2w_{23}+x_3w_{33}+x_4w_{43}+b_3\\
$$

여기에서 o_1은 고양이 점수이고 o_2는 닭 점수, o_3는 개 점수이다.

각 출력은 모든 입력 feature를 사용한다. 이런 층을 fully connected layer, 완전연결층이라고 부른다.

행렬로는 
$$
O=XW+b
$$

배치 크기가 N, feature수가 D, 클래스 수가 C라면
```py
X.shape == [N, D]
W.shape == [D, C]
b.shape == [C]

O = X @ W + b
O.shape == [N, C]

N = 32   # 배치 크기
D = 4    # feature 수
C = 3    # 클래스 수

X.shape == [32, 4]
W.shape == [4, 3]
b.shape == [3]

O.shape == [32, 3]
```

각 행은 샘플 하나이고, 각 열은 클래스 하나다. D2L도 배치 입력을 X∈R^n×d, 가중치를 W∈R^d×q, 출력을 O∈R^n×q로 표현한다고 한다.

## 6. PyTorch의 nn.Linear에서는 shape이 반대처럼 보인다.

PyTorch에선 이렇게 작성한다.

```py
model = nn.Linear(4, 3)
```
입력 feature 4개 => 출력 클래스 점수 3개

하지만 실제 저장된 가중치 shape는 이렇다.

```py
model.weight.shape == [3, 4]
model.bias.shape == [3]
```

PyTorch 내부계산이 아래와 같아 [3, 4]이다.
```py
logits = X @ model.weight.T + model.bias
```

shape을 넣으면 이런식이다.
```text
X                    : [N, 4]
model.weight          : [3, 4]
model.weight.T        : [4, 3]

X @ model.weight.T    : [N, 3]
```

```py
X @ W # 수학에선

X @ weight.T # nnLinear에선
```

이렇다. PyTorch의 nn.Linear는 weight를 [out_features, in_features] 형태로 저장한다고 한다.

## 7. Logit이란 무엇인가

모델이 선형 계산으로 출력한 값 O를 logits라고 한다.

예를 들어서 모델이 아래를 출력했다고 해보자.

```py
logits = [2.0, 1.0, 0.1]
```

의미는 이렇다

고양이 점수: 2.0
닭 점수:     1.0
개 점수:     0.1

가장 높은 값은 고양이 점수 2이므로 모델은 고양이를 가장 유력하다고 본다.

하지만 logits는 확률이 아니다.

합이 1이 아니고, 음수가 나올 수도 있고, 1보다 클 수도 있다.

이런것도 정상적인 logits이다.

```py
logits = [-3.2, 5.7, 1.4]
```
따라서 logits를 확률로 변환하는 과정이 필요하다. D2L은 선형 출력만으로는 값이 음수가 되거나 합이 1이 아닐 수 있기 때문에 확률로 직접 해석할 수 없다고 설명한다.

## 8. Softmax는 logits를 확률로 변환한다.

Softmax공식은 이렇다.
$$
\hat{y}_i=\frac{\exp(o_i)}{\displaystyle\sum_{j=1}^{C}\exp(o_j)}
$$

하는 것은
1. 각 점수에 exp를 적용해서 양수로 만든다.
2. 전체 합으로 나눠서 합이 1이 되게 만든다.

예를 들어서
```py
logtis = [2.0, 1.0, 0.1]
```
먼저 exponential을 적용한다.

```text
exp(2.0) ≈ 7.39
exp(1.0) ≈ 2.72
exp(0.1) ≈ 1.11
```

합은 약 11.22이다. 이걸 각각 합으로 나눈다.

```text
고양이: 7.39 / 11.22 ≈ 0.659
닭:     2.72 / 11.22 ≈ 0.242
개:     1.11 / 11.22 ≈ 0.099
```

```py
probabilities = [0.659, 0.242, 0.099]
```

이러면 모든값이 0 이상이고 모든 값이 1 이하이며 전체 합이 1이다. 확률처럼 해석할 수 있다.

고양이일 확률 약 65.9%, 닭일 확률 약 24.2%, 개일 확률 약 9.9%

Softmax는 각 logit에 exponential을 적용하고 합으로 나눠 확률분포로 만든다.

In [ ]:
import torch

logits = torch.tensor([[2.0, 1.0, 0.1]])
probabilities = torch.softmax(logits, dim=1) # dim=1은 각 샘플 클래스 방향으로 softmax적용
 # Softmax는 지정된 차원의 원소들을 0과 1사이로 변환하고 합이 1이 되도록 정규화함.
print(probabilities)

## 9. 왜 exponential을 사용하나

그냥 모든 숫자를 합으로 나누면 안되나 생각할 수 있다.

예를 들어서
```py
logits = [-2, 1, 3]
```

이걸 합으로 나누면 음수가 남는다. 음수 확률은 존재할 수 없다.
Exponential은 모든 실수를 양수로 바꿔준다.

```text
exp(-2) > 0
exp(1)  > 0
exp(3)  > 0
```

원래 더 큰 점수는 변환 후에도 더 커진다.

```text
3 > 1
exp(3) > exp(1)
```
점수 순서는 유지하면서 확률로 만들 수 있다.